# Homework 6 Solution

This notebook provides one possible solution path for the international finance homework.  
It keeps the same **three-part architecture** as the lecture notebook:

1. obtain market data and compute basic statistics  
2. visualize market behavior  
3. build a Markowitz mean-variance portfolio  

**Important note:** Yahoo Finance is a live online source. Exact values may change slightly if you rerun the notebook on a different date, but the workflow should remain the same.


## Requirements used in this solution

- Watchlist: `SPY`, `VEA`, `EEM`, `AGG`
- Sample period: `2022-01-01` to `2025-01-01`
- Annualization convention: `252` trading days
- Assumed annualized risk-free rate: `0.02`
- Portfolio constraints: long-only, fully invested


In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "outputs").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# If needed in Colab, you can manually override the root like this:
# PROJECT_ROOT = Path("/content/big-data-analysis-course-english")

PROJECT_ROOT


In [ ]:
MODULE_OUTPUT_DIR = OUTPUT_DIR / "homework_06"
MODULE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODULE_OUTPUT_DIR


**Optional setup**


In [ ]:
%pip install -q yfinance scipy seaborn ipywidgets


In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import scipy.optimize as sco

warnings.filterwarnings("ignore")

TRADING_DAYS = 252
RISK_FREE_RATE = 0.02
START_DATE = "2022-01-01"
END_DATE = "2025-01-01"
SIMULATION_SIZE = 5000

def normalize_datetime_column(df: pd.DataFrame, col: str = "date") -> pd.DataFrame:
    df = df.copy()
    df[col] = pd.to_datetime(df[col])
    try:
        df[col] = df[col].dt.tz_localize(None)
    except (TypeError, AttributeError):
        pass
    return df

def get_price_block(downloaded: pd.DataFrame, field: str) -> pd.DataFrame:
    if isinstance(downloaded.columns, pd.MultiIndex):
        block = downloaded.xs(field, axis=1, level=0)
    else:
        block = downloaded[[field]].copy()
    block.columns = [str(c) for c in block.columns]
    return block

def wide_to_long(price_df: pd.DataFrame, value_name: str) -> pd.DataFrame:
    out = price_df.stack(dropna=False).rename(value_name).reset_index()
    out.columns = ["date", "ticker", value_name]
    return out


# Part 1. Obtain market data and compute basic statistics


In [ ]:
institutional_universe = pd.DataFrame(
    {
        "ticker": ["SPY", "VEA", "EEM", "AGG"],
        "instrument_name": [
            "SPDR S&P 500 ETF Trust",
            "Vanguard FTSE Developed Markets ETF",
            "iShares MSCI Emerging Markets ETF",
            "iShares Core U.S. Aggregate Bond ETF",
        ],
        "asset_class": ["Equity", "Equity", "Equity", "Bond"],
        "market_focus": [
            "United States",
            "Developed Markets ex U.S.",
            "Emerging Markets",
            "U.S. Investment-Grade Bonds",
        ],
        "teaching_role": [
            "Core U.S. equity proxy",
            "Developed-market diversification",
            "Emerging-market diversification",
            "Defensive fixed-income allocation",
        ],
    }
)

ticker_list = institutional_universe["ticker"].tolist()
instrument_name_map = dict(zip(institutional_universe["ticker"], institutional_universe["instrument_name"]))

recent_download = yf.download(
    tickers=ticker_list,
    period="5d",
    interval="1d",
    auto_adjust=True,
    progress=False,
    threads=False,
    group_by="column",
)

recent_close = get_price_block(recent_download, "Close")
latest_snapshot = recent_close.ffill().iloc[-1].rename("latest_close").to_frame().reset_index()
latest_snapshot.columns = ["ticker", "latest_close"]
latest_snapshot = latest_snapshot.merge(institutional_universe, on="ticker", how="left")
latest_snapshot.to_csv(MODULE_OUTPUT_DIR / "latest_snapshot.csv", index=False)

latest_snapshot


In [ ]:
multi_download = yf.download(
    tickers=ticker_list,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False,
    threads=False,
    group_by="column",
)

close_prices = get_price_block(multi_download, "Close")
open_prices = get_price_block(multi_download, "Open")

close_long = wide_to_long(close_prices, "close")
open_long = wide_to_long(open_prices, "open")

market_df = (
    close_long
    .merge(open_long, on=["date", "ticker"], how="left")
    .dropna(subset=["close"])
)

market_df = normalize_datetime_column(market_df, "date")
market_df["daily_return"] = market_df.groupby("ticker")["close"].pct_change()
market_df = market_df.merge(institutional_universe, on="ticker", how="left")

market_df.to_csv(MODULE_OUTPUT_DIR / "market_data_long_format.csv", index=False)
market_df.head()


## Asset-level annualized statistics

Annualized return = mean daily return × 252  
Annualized volatility = standard deviation of daily return × √252  
Sharpe ratio = (annualized return − risk-free rate) / annualized volatility


In [ ]:
asset_metrics = (
    market_df
    .dropna(subset=["daily_return"])
    .groupby(["ticker", "instrument_name", "asset_class"])["daily_return"]
    .agg(["mean", "std"])
    .reset_index()
)

asset_metrics["annual_return"] = asset_metrics["mean"] * TRADING_DAYS
asset_metrics["annual_volatility"] = asset_metrics["std"] * np.sqrt(TRADING_DAYS)
asset_metrics["sharpe_ratio"] = (
    (asset_metrics["annual_return"] - RISK_FREE_RATE) / asset_metrics["annual_volatility"]
)

asset_metrics = asset_metrics[
    ["ticker", "instrument_name", "asset_class", "annual_return", "annual_volatility", "sharpe_ratio"]
].sort_values("sharpe_ratio", ascending=False)

asset_metrics = asset_metrics.round(4)
asset_metrics.to_csv(MODULE_OUTPUT_DIR / "asset_metrics.csv", index=False)
asset_metrics


In [ ]:
summary_table = (
    market_df.groupby("instrument_name")[["daily_return", "close"]]
    .describe()
    .round(4)
    .T
)

summary_table.to_csv(MODULE_OUTPUT_DIR / "grouped_summary_table.csv")
summary_table


# Part 2. Visualize market data


In [ ]:
spy = market_df[market_df["ticker"] == "SPY"].copy()

plt.figure(figsize=(15, 5))
plt.plot(spy["date"], spy["open"], linestyle="-.", linewidth=1, marker=".", alpha=0.5)
plt.plot(spy["date"], spy["close"])
plt.xlabel("Date")
plt.ylabel("Price")
plt.title("SPY: Open vs. Close")
plt.legend(["Opening price", "Closing price"], loc="best")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "homework6_spy_open_close.png", bbox_inches="tight")
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(data=market_df, x="date", y="daily_return", hue="instrument_name")
plt.title("Daily Return Series")
plt.xlabel("Date")
plt.ylabel("Daily return")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "homework6_daily_return_lineplot.png", bbox_inches="tight")
plt.show()


In [ ]:
market_df = market_df.sort_values(["ticker", "date"]).copy()

market_df["cumulative_return"] = (
    market_df.groupby("ticker")["daily_return"]
    .transform(lambda s: (1 + s.fillna(0)).cumprod() - 1)
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=market_df, x="date", y="cumulative_return", hue="instrument_name")
plt.title("Compounded Cumulative Return")
plt.xlabel("Date")
plt.ylabel("Cumulative return")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "homework6_cumulative_return.png", bbox_inches="tight")
plt.show()


A useful teaching note here is that **compounded** cumulative returns are more appropriate than a simple cumulative sum of daily returns.


# Part 3. Markowitz mean-variance portfolio


In [ ]:
returns = (
    market_df[["ticker", "daily_return", "date"]]
    .pivot_table(values="daily_return", index="date", columns="ticker")
    .dropna(how="all")
    .dropna()
)

returns.head()


In [ ]:
noa = len(ticker_list)
bounds = ((0, 1),) * noa
constraints = {"type": "eq", "fun": lambda x: np.sum(x) - 1}

def statistics(weights):
    weights = np.array(weights)
    port_return = np.sum(returns.mean() * TRADING_DAYS * weights)
    port_volatility = np.sqrt(np.dot(weights.T, np.dot(returns.cov() * TRADING_DAYS, weights)))
    sharpe = (port_return - RISK_FREE_RATE) / port_volatility
    return np.array([port_return, port_volatility, sharpe])

def negative_sharpe(weights):
    return -statistics(weights)[2]

def portfolio_vol(weights):
    return statistics(weights)[1]

np.random.seed(42)

portfolio_returns = []
portfolio_volatility = []
portfolio_sharpe = []
portfolio_weights = []

for _ in range(SIMULATION_SIZE):
    weights = np.random.random(noa)
    weights /= np.sum(weights)

    stats = statistics(weights)
    portfolio_returns.append(stats[0])
    portfolio_volatility.append(stats[1])
    portfolio_sharpe.append(stats[2])
    portfolio_weights.append(weights)

portfolio_returns = np.array(portfolio_returns)
portfolio_volatility = np.array(portfolio_volatility)
portfolio_sharpe = np.array(portfolio_sharpe)

monte_carlo_df = pd.DataFrame(
    {
        "expected_return": portfolio_returns,
        "expected_volatility": portfolio_volatility,
        "sharpe_ratio": portfolio_sharpe,
    }
)
monte_carlo_df.to_csv(MODULE_OUTPUT_DIR / "monte_carlo_portfolios.csv", index=False)

plt.figure(figsize=(8, 4))
plt.scatter(portfolio_volatility, portfolio_returns, c=portfolio_sharpe, marker="o")
plt.xlabel("Expected volatility")
plt.ylabel("Expected return")
plt.title("Monte Carlo Portfolio Simulation")
plt.grid(True)
plt.colorbar(label="Sharpe ratio")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "homework6_monte_carlo_portfolios.png", bbox_inches="tight")
plt.show()


In [ ]:
opt_sharpe = sco.minimize(
    negative_sharpe,
    noa * [1.0 / noa],
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
)

max_sharpe_weights = pd.DataFrame(
    {
        "ticker": ticker_list,
        "instrument_name": [instrument_name_map[t] for t in ticker_list],
        "weight": opt_sharpe.x,
    }
).sort_values("weight", ascending=False)

max_sharpe_weights["weight"] = max_sharpe_weights["weight"].round(6)
max_sharpe_weights.to_csv(MODULE_OUTPUT_DIR / "max_sharpe_weights.csv", index=False)

max_sharpe_stats = statistics(opt_sharpe.x)
print(f"Maximum-Sharpe expected return: {max_sharpe_stats[0]:.4f}")
print(f"Maximum-Sharpe expected volatility: {max_sharpe_stats[1]:.4f}")
print(f"Maximum-Sharpe Sharpe ratio: {max_sharpe_stats[2]:.4f}")

max_sharpe_weights


In [ ]:
opt_min_vol = sco.minimize(
    portfolio_vol,
    noa * [1.0 / noa],
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
)

min_vol_weights = pd.DataFrame(
    {
        "ticker": ticker_list,
        "instrument_name": [instrument_name_map[t] for t in ticker_list],
        "weight": opt_min_vol.x,
    }
).sort_values("weight", ascending=False)

min_vol_weights["weight"] = min_vol_weights["weight"].round(6)
min_vol_weights.to_csv(MODULE_OUTPUT_DIR / "min_vol_weights.csv", index=False)

min_vol_stats = statistics(opt_min_vol.x)
print(f"Minimum-volatility expected return: {min_vol_stats[0]:.4f}")
print(f"Minimum-volatility expected volatility: {min_vol_stats[1]:.4f}")
print(f"Minimum-volatility Sharpe ratio: {min_vol_stats[2]:.4f}")

min_vol_weights


In [ ]:
target_returns = np.linspace(
    (returns.mean() * TRADING_DAYS).min(),
    (returns.mean() * TRADING_DAYS).max(),
    100,
)

target_volatility = []

for target in target_returns:
    target_constraints = (
        {"type": "eq", "fun": lambda x, target=target: statistics(x)[0] - target},
        {"type": "eq", "fun": lambda x: np.sum(x) - 1},
    )

    result = sco.minimize(
        portfolio_vol,
        noa * [1.0 / noa],
        method="SLSQP",
        bounds=bounds,
        constraints=target_constraints,
    )

    target_volatility.append(result["fun"])

efficient_frontier_df = pd.DataFrame(
    {"target_return": target_returns, "target_volatility": target_volatility}
)
efficient_frontier_df.to_csv(MODULE_OUTPUT_DIR / "efficient_frontier_points.csv", index=False)

plt.figure(figsize=(8, 4))
plt.scatter(portfolio_volatility, portfolio_returns, c=portfolio_sharpe, marker="o", alpha=0.7)
plt.scatter(target_volatility, target_returns, c=np.array(target_returns) / np.array(target_volatility), cmap="autumn", marker="x", s=8)
plt.plot(max_sharpe_stats[1], max_sharpe_stats[0], "r*", markersize=15)
plt.plot(min_vol_stats[1], min_vol_stats[0], "y*", markersize=15)
plt.xlabel("Expected volatility")
plt.ylabel("Expected return")
plt.title("Efficient Frontier")
plt.grid(True)
plt.colorbar(label="Return / volatility")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "homework6_efficient_frontier.png", bbox_inches="tight")
plt.show()


In [ ]:
portfolio_summary = pd.DataFrame(
    {
        "portfolio": ["maximum_sharpe", "minimum_volatility"],
        "expected_return": [max_sharpe_stats[0], min_vol_stats[0]],
        "expected_volatility": [max_sharpe_stats[1], min_vol_stats[1]],
        "sharpe_ratio": [max_sharpe_stats[2], min_vol_stats[2]],
    }
).round(4)

portfolio_summary.to_csv(MODULE_OUTPUT_DIR / "portfolio_summary.csv", index=False)
portfolio_summary


## Example interpretation

Because this homework uses live market data, the exact ranking and weights may change slightly when you rerun it.  
A good answer should still explain the **direction** of the results clearly:

- identify the ETF with the best sample risk-adjusted performance,
- compare the risk profile of the maximum-Sharpe and minimum-volatility portfolios,
- and explain how the efficient frontier illustrates diversification benefits.


In [ ]:
top_asset = asset_metrics.iloc[0]["instrument_name"]
top_asset_sharpe = asset_metrics.iloc[0]["sharpe_ratio"]

asset_class_map = institutional_universe.set_index("ticker")["asset_class"].to_dict()

max_sharpe_equity_share = (
    max_sharpe_weights.assign(asset_class=lambda d: d["ticker"].map(asset_class_map))
    .query("asset_class == 'Equity'")["weight"]
    .sum()
)

min_vol_equity_share = (
    min_vol_weights.assign(asset_class=lambda d: d["ticker"].map(asset_class_map))
    .query("asset_class == 'Equity'")["weight"]
    .sum()
)

print(f"In this run, the highest sample Sharpe ratio belongs to {top_asset} ({top_asset_sharpe:.4f}).")
print(f"Equity share in the maximum-Sharpe portfolio: {max_sharpe_equity_share:.4f}")
print(f"Equity share in the minimum-volatility portfolio: {min_vol_equity_share:.4f}")
print("The efficient frontier shows the best return-volatility combinations available from this ETF universe.")
